# Phase 1–2: phenotype, data readiness, and cohort review

This notebook audits the available NIS variables and constructs the cached adult hematologic-malignancy hospitalization cohort. The phenotype is **draft pending clinical review**. No adjusted modeling is performed here.

Choose **Run → Run All Cells**. The first run builds the local cohort; unchanged reruns use the cache.

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import Markdown, display

repo_root = Path.cwd()
if not (repo_root / 'src').is_dir():
    repo_root = repo_root.parent
if not (repo_root / 'src/phase_1_2.py').is_file():
    raise RuntimeError('Open this notebook from the IPC-MPC-Study repository.')
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

from src.phase_1_2 import main
result = main([])

{
  "phenotype_version": "0.1-draft",
  "phenotype_status": "Draft for investigator and clinical code review; not frozen",
  "dataset_files": 144,
  "cache_used": true,
  "unweighted_hm_discharges": 994992,
  "weighted_hm_discharges_2016_2022": 4974958.0,
  "unweighted_with_sepsis": 158421,
  "weighted_with_sepsis": 792105.0,
  "unweighted_sepsis_percent": 15.92,
  "weighted_sepsis_percent": 15.92,
  "unweighted_with_palliative_care": 80733,
  "weighted_with_palliative_care": 403665.0,
  "weighted_palliative_care_percent": 8.11,
  "unweighted_with_multiple_hm_groups": 44350,
  "weighted_with_multiple_hm_groups": 221750.0,
  "weighted_multiple_hm_groups_percent": 4.46,
  "output_directory": "/Users/evelynzhang/Desktop/dev/IPC-MPC-Study/outputs/phase_1_2"
}


## Phase 1: data-readiness audit

In [2]:
output_dir = repo_root / 'outputs/phase_1_2'
audit = pd.read_csv(output_dir / 'data_readiness_audit.csv')
display(audit)
print('Copy/paste CSV:\n' + audit.to_csv(index=False))

,component,status,detail
0,Diagnosis fields,READY,40 fields available
1,Adult cohort,READY,Requires AGE
2,Discharge weights,READY,Requires DISCWT
3,Survey strata,READY,Requires NIS_STRATUM
4,Mortality,READY,Requires DIED
5,Length of stay,READY,Requires LOS
6,Hospital characteristics,DERIVED/REVIEW,"Division, region, control, location/teaching, ..."
7,Mechanical ventilation,OUT OF SCOPE,Removed from the requested analysis; procedure...


Copy/paste CSV:
component,status,detail
Diagnosis fields,READY,40 fields available
Adult cohort,READY,Requires AGE
Discharge weights,READY,Requires DISCWT
Survey strata,READY,Requires NIS_STRATUM
Mortality,READY,Requires DIED
Length of stay,READY,Requires LOS
Hospital characteristics,DERIVED/REVIEW,"Division, region, control, location/teaching, and bed size decoded from four-digit NIS_STRATUM"
Mechanical ventilation,OUT OF SCOPE,Removed from the requested analysis; procedure fields are not required



## Decisions required before freezing the phenotype

In [3]:
review = json.loads((output_dir / 'phenotype_review.json').read_text())
display(Markdown(f"**Version:** {review['version']}  \n**Status:** {review['review_status']}"))
display(Markdown('\n'.join(f"{i}. {item}" for i, item in enumerate(review['decisions_required_before_freeze'], 1))))

**Version:** 0.1-draft  
**Status:** Draft for investigator and clinical code review; not frozen

1. Clinically verify every lymphoma exclusion, especially suffix-A/site codes.
2. Decide whether remission-coded hematologic malignancies are excluded consistently across all groups.
3. Decide how CLL/SLL coding overlap should be classified.
4. Decide whether A41-only sepsis is primary and whether a broader explicit-sepsis definition is a sensitivity analysis.
5. Confirm whether first-listed qualifying HM diagnosis is the primary subtype rule.

In [4]:
phenotype = json.loads((repo_root / 'config/hm_phenotype_v0_1.json').read_text())
rules = pd.DataFrame([
    {
        'subtype': item['label'],
        'include_prefixes': ', '.join(item['include_prefixes']),
        'excluded_exact_codes': ', '.join(item.get('exclude_exact', [])) or 'None',
        'review_note': item.get('review_note', ''),
    }
    for item in phenotype['subtypes']
])
display(Markdown('### Draft HM subtype rules'))
display(rules)
print('Copy/paste CSV:\n' + rules.to_csv(index=False))

### Draft HM subtype rules

,subtype,include_prefixes,excluded_exact_codes,review_note
0,Lymphoma,"C81, C82, C83, C84, C85, C86, C880, C884","C810A, C811A, C812A, C813A, C814A, C817A, C819...",Several suffix-A and site-specific exclusions ...
1,Acute myeloid leukemia,"C920, C926, C92A, C924, C925, C92Z, C929","C9201, C9261, C92A1, C9241, C9251, C92Z1, C9291",
2,Chronic myeloid leukemia,"C921, C922","C9211, C9221",
3,CLL/chronic leukemia,"C911, C951",C9511,Confirm whether C91.11 (in remission) should a...
4,Acute lymphoblastic/unspecified acute leukemia,"C910, C950","C9101, C9501",
5,Other leukemia,"C913, C915, C916, C91A, C91Z, C919, C930, C931...","C9131, C9151, C9161, C91A1, C91Z1, C9191, C930...",
6,Multiple myeloma/plasma-cell neoplasm,C90,"C9001, C9011, C9021, C9031",
7,Myelodysplastic disease,"C946, D46",None,
8,Myeloproliferative neoplasm,"D45, D473, D474, D7581",None,


Copy/paste CSV:
subtype,include_prefixes,excluded_exact_codes,review_note
Lymphoma,"C81, C82, C83, C84, C85, C86, C880, C884","C810A, C811A, C812A, C813A, C814A, C817A, C819A, C820A, C821A, C822A, C823A, C824A, C825A, C826A, C828A, C829A, C830A, C831A, C833A, C835A, C837A, C838A, C839A, C840A, C841A, C844A, C846A, C847B, C84AA, C84ZA, C849A, C851A, C852A, C858A, C859A, C8601, C8611, C8621, C8631, C8641, C8651, C8661, C8801, C8841, C9111, C9141",Several suffix-A and site-specific exclusions require clinical verification before freezing.
Acute myeloid leukemia,"C920, C926, C92A, C924, C925, C92Z, C929","C9201, C9261, C92A1, C9241, C9251, C92Z1, C9291",
Chronic myeloid leukemia,"C921, C922","C9211, C9221",
CLL/chronic leukemia,"C911, C951",C9511,Confirm whether C91.11 (in remission) should also be excluded and how CLL/SLL overlap should be handled.
Acute lymphoblastic/unspecified acute leukemia,"C910, C950","C9101, C9501",
Other leukemia,"C913, C915, C916, C91A, C91Z, C919, C930, C931, C9

## Phase 2: adult HM cohort results

Counts refer to inpatient discharge records, not unique patients. Weighted totals covering all seven years are cumulative national discharge estimates, not annual counts.

In [5]:
summary = json.loads((output_dir / 'cohort_summary.json').read_text())
summary_table = pd.DataFrame({'measure': list(summary), 'value': list(summary.values())})
display(summary_table)
print('Copy/paste CSV:\n' + summary_table.to_csv(index=False))

,measure,value
0,unweighted_hm_discharges,994992.00
1,weighted_hm_discharges_2016_2022,4974958.00
2,unweighted_with_sepsis,158421.00
3,weighted_with_sepsis,792105.00
4,unweighted_sepsis_percent,15.92
5,weighted_sepsis_percent,15.92
6,unweighted_with_palliative_care,80733.00
7,weighted_with_palliative_care,403665.00
8,weighted_palliative_care_percent,8.11
9,unweighted_with_multiple_hm_groups,44350.00


Copy/paste CSV:
measure,value
unweighted_hm_discharges,994992.0
weighted_hm_discharges_2016_2022,4974958.0
unweighted_with_sepsis,158421.0
weighted_with_sepsis,792105.0
unweighted_sepsis_percent,15.92
weighted_sepsis_percent,15.92
unweighted_with_palliative_care,80733.0
weighted_with_palliative_care,403665.0
weighted_palliative_care_percent,8.11
unweighted_with_multiple_hm_groups,44350.0
weighted_with_multiple_hm_groups,221750.0
weighted_multiple_hm_groups_percent,4.46



In [6]:
by_year = pd.read_csv(output_dir / 'cohort_by_year.csv')
display(Markdown('### Cohort by year'))
display(by_year)
print('Copy/paste CSV:\n' + by_year.to_csv(index=False))

### Cohort by year

,year,unweighted_hm_discharges,weighted_hm_discharges,unweighted_with_sepsis,weighted_sepsis_percent,unweighted_with_palliative_care,weighted_palliative_care_percent
0,2016,144171,720855.0,21442,14.87,9738,6.75
1,2017,143704,718520.0,21734,15.12,10435,7.26
2,2018,146473,732365.0,23098,15.77,11127,7.60
3,2019,153047,765235.0,24411,15.95,11817,7.72
4,2020,140796,703980.0,23786,16.89,12071,8.57
5,2021,141380,706899.0,23375,16.53,13119,9.28
6,2022,125421,627105.0,20575,16.40,12426,9.91


Copy/paste CSV:
year,unweighted_hm_discharges,weighted_hm_discharges,unweighted_with_sepsis,weighted_sepsis_percent,unweighted_with_palliative_care,weighted_palliative_care_percent
2016,144171,720855.0,21442,14.87,9738,6.75
2017,143704,718520.0,21734,15.12,10435,7.26
2018,146473,732365.0,23098,15.77,11127,7.6
2019,153047,765235.0,24411,15.95,11817,7.72
2020,140796,703980.0,23786,16.89,12071,8.57
2021,141380,706899.0,23375,16.53,13119,9.28
2022,125421,627105.0,20575,16.4,12426,9.91



In [7]:
by_subtype = pd.read_csv(output_dir / 'cohort_by_subtype.csv')
display(Markdown('### First-listed mutually exclusive HM subtype'))
display(by_subtype)
print('Copy/paste CSV:\n' + by_subtype.to_csv(index=False))

### First-listed mutually exclusive HM subtype

,hm_subtype,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,unweighted_with_sepsis,weighted_sepsis_percent,unweighted_with_palliative_care,weighted_palliative_care_percent
0,lymphoma,301494,1507470.0,42749,14.18,23491,7.79
1,mpn,170226,851130.0,31995,18.80,8606,5.06
2,myeloma_plasma_cell,150044,750220.0,22096,14.73,13580,9.05
3,cll_chronic_leukemia,108837,544185.0,17595,16.17,8272,7.60
4,mds,106324,531620.0,16745,15.75,9809,9.23
5,aml,79382,396910.0,15820,19.93,10984,13.84
6,all,32145,160725.0,4348,13.53,2200,6.84
7,cml,26540,132700.0,3393,12.78,1760,6.63
8,other_leukemia,20000,100000.0,3680,18.40,2031,10.16


Copy/paste CSV:
hm_subtype,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,unweighted_with_sepsis,weighted_sepsis_percent,unweighted_with_palliative_care,weighted_palliative_care_percent
lymphoma,301494,1507470.0,42749,14.18,23491,7.79
mpn,170226,851130.0,31995,18.8,8606,5.06
myeloma_plasma_cell,150044,750220.0,22096,14.73,13580,9.05
cll_chronic_leukemia,108837,544185.0,17595,16.17,8272,7.6
mds,106324,531620.0,16745,15.75,9809,9.23
aml,79382,396910.0,15820,19.93,10984,13.84
all,32145,160725.0,4348,13.53,2200,6.84
cml,26540,132700.0,3393,12.78,1760,6.63
other_leukemia,20000,100000.0,3680,18.4,2031,10.16



In [8]:
overlap = pd.read_csv(output_dir / 'cohort_overlap.csv')
display(Markdown('### Number of distinct HM groups per hospitalization'))
display(overlap)
print('Copy/paste CSV:\n' + overlap.to_csv(index=False))

### Number of distinct HM groups per hospitalization

,hm_group_count,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,weighted_percent
0,1,950642,4753208.0,95.54
1,2,42703,213515.0,4.29
2,3,1600,8000.0,0.16
3,4+,47,235.0,0.00


Copy/paste CSV:
hm_group_count,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,weighted_percent
1,950642,4753208.0,95.54
2,42703,213515.0,4.29
3,1600,8000.0,0.16
4+,47,235.0,0.0



## Stratum-derived hospital characteristics

For 2016–2022, the four digits of `NIS_STRATUM` encode Census division, ownership/control, location/teaching status, and bed-size category. Region is calculated from division. These are **stratum-derived characteristics**: HCUP notes that collapsed sampling strata can occasionally differ from a hospital's separate actual-characteristic variables.

In [9]:
hospital_files = {
    'Region': 'hospital_by_region.csv',
    'Census division': 'hospital_by_division.csv',
    'Location/teaching': 'hospital_by_location_teaching.csv',
    'Bed size': 'hospital_by_bed_size.csv',
    'Ownership/control': 'hospital_by_control.csv',
}
for title, filename in hospital_files.items():
    table = pd.read_csv(output_dir / filename)
    display(Markdown(f'### {title}'))
    display(table)
    print('Copy/paste CSV:\n' + table.to_csv(index=False))
validation = pd.read_csv(output_dir / 'stratum_decode_validation.csv')
display(Markdown('### Decode validation'))
display(validation)

### Region

,hospital_region,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,weighted_percent
0,South,362561,1812805.0,36.44
1,Midwest,229923,1149615.0,23.11
2,Northeast,205522,1027610.0,20.66
3,West,196986,984929.0,19.80


Copy/paste CSV:
hospital_region,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,weighted_percent
South,362561,1812805.0,36.44
Midwest,229923,1149615.0,23.11
Northeast,205522,1027610.0,20.66
West,196986,984929.0,19.8



### Census division

,hospital_division_code,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,weighted_percent
0,5,203497,1017485.0,20.45
1,3,157398,786990.0,15.82
2,2,151681,758405.0,15.24
3,9,138473,692365.0,13.92
4,7,102544,512720.0,10.31
5,4,72525,362625.0,7.29
6,8,58513,292564.0,5.88
7,6,56520,282600.0,5.68
8,1,53841,269205.0,5.41


...

### Ownership/control

,hospital_control,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,weighted_percent
0,"Private, not-for-profit",765651,3828254.0,76.95
1,"Government, nonfederal",126985,634924.0,12.76
2,"Private, investor-owned",102356,511780.0,10.29


Copy/paste CSV:
hospital_control,unweighted_hm_discharges,weighted_hm_discharges_2016_2022,weighted_percent
"Private, not-for-profit",765651,3828254.0,76.95
"Government, nonfederal",126985,634924.0,12.76
"Private, investor-owned",102356,511780.0,10.29



### Decode validation

,unweighted_hm_discharges,invalid_or_unknown_decodes
0,994992,0


## Review checkpoint

Before proceeding, review whether cohort size, yearly stability, subtype distribution, overlap frequency, sepsis prevalence, and palliative-care prevalence are clinically plausible. Phase 3 or later modeling should wait until the phenotype decisions above are resolved.

## Consolidated review tables

These tables summarize the currently computed Phase 1–2 data. Weighted counts spanning 2016–2022 are cumulative national discharge estimates. They are not counts of unique patients.

In [10]:
table_1 = pd.read_csv(output_dir / 'review_table_1_cohort_overview.csv')
display(Markdown('### Table 1. Adult hematologic-malignancy cohort overview'))
display(table_1)
print('Copy/paste CSV:\n' + table_1.to_csv(index=False))

### Table 1. Adult hematologic-malignancy cohort overview

,cohort_measure,unweighted_n,weighted_n_2016_2022,weighted_percent
0,All adult HM discharges,994992,4974958.0,100.00
1,Documented sepsis (A41*),158421,792105.0,15.92
2,Documented inpatient palliative-care use (Z51.5),80733,403665.0,8.11
3,Multiple HM groups,44350,221750.0,4.46


Copy/paste CSV:
cohort_measure,unweighted_n,weighted_n_2016_2022,weighted_percent
All adult HM discharges,994992,4974958.0,100.0
Documented sepsis (A41*),158421,792105.0,15.92
Documented inpatient palliative-care use (Z51.5),80733,403665.0,8.11
Multiple HM groups,44350,221750.0,4.46



In [11]:
table_2 = pd.read_csv(output_dir / 'cohort_by_year.csv').rename(columns={
    'unweighted_hm_discharges': 'unweighted_n',
    'weighted_hm_discharges': 'weighted_n',
    'unweighted_with_sepsis': 'sepsis_unweighted_n',
    'weighted_sepsis_percent': 'sepsis_weighted_percent',
    'unweighted_with_palliative_care': 'palliative_care_unweighted_n',
    'weighted_palliative_care_percent': 'palliative_care_weighted_percent',
})
display(Markdown('### Table 2. Cohort, sepsis, and palliative-care documentation by year'))
display(table_2)
print('Copy/paste CSV:\n' + table_2.to_csv(index=False))

### Table 2. Cohort, sepsis, and palliative-care documentation by year

,year,unweighted_n,weighted_n,sepsis_unweighted_n,sepsis_weighted_percent,palliative_care_unweighted_n,palliative_care_weighted_percent
0,2016,144171,720855.0,21442,14.87,9738,6.75
1,2017,143704,718520.0,21734,15.12,10435,7.26
2,2018,146473,732365.0,23098,15.77,11127,7.60
3,2019,153047,765235.0,24411,15.95,11817,7.72
4,2020,140796,703980.0,23786,16.89,12071,8.57
5,2021,141380,706899.0,23375,16.53,13119,9.28
6,2022,125421,627105.0,20575,16.40,12426,9.91


Copy/paste CSV:
year,unweighted_n,weighted_n,sepsis_unweighted_n,sepsis_weighted_percent,palliative_care_unweighted_n,palliative_care_weighted_percent
2016,144171,720855.0,21442,14.87,9738,6.75
2017,143704,718520.0,21734,15.12,10435,7.26
2018,146473,732365.0,23098,15.77,11127,7.6
2019,153047,765235.0,24411,15.95,11817,7.72
2020,140796,703980.0,23786,16.89,12071,8.57
2021,141380,706899.0,23375,16.53,13119,9.28
2022,125421,627105.0,20575,16.4,12426,9.91



In [12]:
table_3 = pd.read_csv(output_dir / 'review_table_3_subtypes.csv')
display(Markdown('### Table 3. Cohort by first-listed HM subtype'))
display(table_3)
print('Copy/paste CSV:\n' + table_3.to_csv(index=False))

### Table 3. Cohort by first-listed HM subtype

,hm_subtype,unweighted_n,weighted_n_2016_2022,sepsis_unweighted_n,sepsis_weighted_percent,palliative_care_unweighted_n,palliative_care_weighted_percent
0,Lymphoma,301494,1507470.0,42749,14.18,23491,7.79
1,Myeloproliferative neoplasm,170226,851130.0,31995,18.80,8606,5.06
2,Multiple myeloma/plasma-cell neoplasm,150044,750220.0,22096,14.73,13580,9.05
3,CLL/chronic leukemia,108837,544185.0,17595,16.17,8272,7.60
4,Myelodysplastic disease,106324,531620.0,16745,15.75,9809,9.23
5,Acute myeloid leukemia,79382,396910.0,15820,19.93,10984,13.84
6,Acute lymphoblastic/unspecified acute leukemia,32145,160725.0,4348,13.53,2200,6.84
7,Chronic myeloid leukemia,26540,132700.0,3393,12.78,1760,6.63
8,Other leukemia,20000,100000.0,3680,18.40,2031,10.16


Copy/paste CSV:
hm_subtype,unweighted_n,weighted_n_2016_2022,sepsis_unweighted_n,sepsis_weighted_percent,palliative_care_unweighted_n,palliative_care_weighted_percent
Lymphoma,301494,1507470.0,42749,14.18,23491,7.79
Myeloproliferative neoplasm,170226,851130.0,31995,18.8,8606,5.06
Multiple myeloma/plasma-cell neoplasm,150044,750220.0,22096,14.73,13580,9.05
CLL/chronic leukemia,108837,544185.0,17595,16.17,8272,7.6
Myelodysplastic disease,106324,531620.0,16745,15.75,9809,9.23
Acute myeloid leukemia,79382,396910.0,15820,19.93,10984,13.84
Acute lymphoblastic/unspecified acute leukemia,32145,160725.0,4348,13.53,2200,6.84
Chronic myeloid leukemia,26540,132700.0,3393,12.78,1760,6.63
Other leukemia,20000,100000.0,3680,18.4,2031,10.16

